# Chapter 1 — When Models Make Things Up

**Book alignment:** Hallucination From First Principles, Chapter 1

**Question this notebook isolates:** Does a scoring rule that rewards guessing change the optimal answer-vs-abstain policy relative to one that penalizes wrong answers?

Synthetic fixtures in this notebook demonstrate the mechanism type only and do not reproduce the book's 10k-row empirical run.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)


## 1. The payoff decides whether guessing is rational

Chapter 1 (sec. 5) contrasts a benchmark payoff (correct +1, wrong 0, abstain 0) with a production payoff (correct +1, abstain 0, confident-wrong −5). We compute the expected value of answering at several correctness probabilities and read off the optimal policy under each rule.


In [ ]:
def ev_answer(p, reward=1.0, wrong_cost=0.0):
    """Expected value of answering with correctness probability p."""
    return p * reward + (1 - p) * (-wrong_cost)


def optimal_action(p, reward=1.0, wrong_cost=0.0):
    return "answer" if ev_answer(p, reward, wrong_cost) > 0 else "abstain"


benchmark = {"reward": 1.0, "wrong_cost": 0.0}  # correct +1 / wrong 0 / abstain 0
production = {"reward": 1.0, "wrong_cost": 5.0}  # correct +1 / abstain 0 / wrong -5

probs = [0.10, 0.30, 0.60, 0.85, 0.95]
print(f"{'p(correct)':>10} | {'bench EV':>8} {'act':>7} | {'prod EV':>8} {'act':>7}")
for p in probs:
    eb = ev_answer(p, **benchmark)
    ep = ev_answer(p, **production)
    print(f"{p:10.2f} | {eb:8.3f} {optimal_action(p, **benchmark):>7} | {ep:8.3f} {optimal_action(p, **production):>7}")

threshold = production["wrong_cost"] / (1.0 + production["wrong_cost"])
print("\nproduction answer threshold p* =", round(threshold, 4))


In [ ]:
assert optimal_action(0.30, **benchmark) == "answer"
assert optimal_action(0.30, **production) == "abstain"
assert optimal_action(0.90, **production) == "answer"
assert optimal_action(0.10, **benchmark) == "answer"  # any p > 0 favors guessing under benchmark
assert abs(threshold - 5 / 6) < 1e-9
print("payoff flip confirmed: p=0.30 -> benchmark=answer, production=abstain")


## 2. Greedy decoding can still hallucinate

Chapter 1 (sec. 3) gives two toy next-token distributions: one where the top token is right (Paris 0.62) and one where the top token is an invented name (0.41) above "I don't know" (0.09). Greedy decoding picks the top in both cases, so determinism does not remove uncertainty about support.


In [ ]:
dist_known = [("Paris", 0.62), ("Lyon", 0.17), ("London", 0.11), ("Berlin", 0.06), ("other", 0.04)]
dist_unknown = [("invented_name_A", 0.41), ("invented_name_B", 0.27), ("I don't know", 0.09), ("other", 0.23)]


def greedy(dist):
    return max(dist, key=lambda kv: kv[1])[0]


def sample_tokens(dist, rng, n):
    toks = [t for t, _ in dist]
    ps = np.array([p for _, p in dist])
    return list(rng.choice(toks, size=n, p=ps))


g_known = greedy(dist_known)
g_unknown = greedy(dist_unknown)
print("greedy known:  ", g_known)
print("greedy unknown:", g_unknown)

s_known = sample_tokens(dist_known, rng, 50)
s_unknown = sample_tokens(dist_unknown, rng, 200)
print("known samples distinct:", sorted(set(s_known)))
frac_invented = sum(t.startswith("invented") for t in s_unknown) / len(s_unknown)
print("unknown: empirical P(invented A or B) =", round(frac_invented, 3))


In [ ]:
assert g_known == "Paris"
assert g_unknown == "invented_name_A"  # deterministic hallucination: wrong continuation is on top
assert g_unknown != "I don't know"
assert len(set(s_known)) > 1  # sampling reveals reachable alternatives on the known question
assert sum(t.startswith("invented") for t in s_unknown) / len(s_unknown) > 0.5
print("deterministic greedy hallucination confirmed; sampling variability confirmed")


## What we earned

- Under the benchmark payoff any p > 0 favors guessing; under the production payoff (wrong −5) the answer threshold is p* = 0.833, so p = 0.30 flips from answer to abstain. The same behavior looks excellent under one metric and reckless under another.
- Greedy decoding is confidently wrong on the unknown distribution (top token invented_name_A at 0.41 beats "I don't know" at 0.09): repeatability is not truth.

Next: Chapter 2 — Hallucination Is Not One Thing, which replaces the single failure word with typed per-claim failure records.
